# Overlap-add and STFT framing

This notebook is the slower companion to `notes/overlap-add-and-stft-framing.md`.

The narrow question is the useful one: once a window is reused frame after frame, how flat is the raw overlap-add sum for a chosen hop?


## The quantity being measured

For hop `H`, the periodic overlap-add profile is

```text
s[p] = Σ_k w[p + kH]
```

with `p` running over one hop period.
If `s[p]` is constant, the raw overlap sum is flat.
If it ripples, some sample phases get a larger aggregate weight than others.


This repo measures that with the **symmetric** window tables already used everywhere else, at frame length `N = 128`.
That is deliberate: the sidecar is about what this repo's actual windows do, not about repeating a periodic-window textbook identity out of context.


In [ ]:
import csv
from pathlib import Path

rows = list(csv.DictReader((Path('..') / 'art' / 'window-overlap-add-metrics.csv').open()))
half_hop = [row for row in rows if row['hop'] == '64']
[(row['name'], round(float(row['max_deviation_pct']), 3)) for row in half_hop]


The half-hop table is the first important surprise if you came in thinking only about spectra:

- rectangular is perfectly flat on raw overlap-add and still awful for leakage
- Hann and Hamming stay close to flat
- Blackman and Kaiser need a smaller hop
- flat-top is not remotely a lazy half-hop choice in this symmetric-table experiment

So overlap-add flatness and spectral honesty are related practical concerns, but they are not the same metric.


In [ ]:
quarter_hop = [row for row in rows if row['hop'] == '32']
[(row['name'], round(float(row['max_deviation_pct']), 4)) for row in quarter_hop]


Quarter hop is the useful middle regime.
Most windows become effectively flat, but flat-top still carries a visible overlap penalty.
That is the second bill after its already-large ENBW.


## Takeaway

This sidecar is really about framing judgment:

1. the window family affects the spectrum you read
2. the hop size affects whether the frame train stays evenly weighted
3. those two choices belong in the same conversation

That is why `spectral-window-lab` now needs the overlap-add lane next to the FFT lane.
